In [35]:
# Import the required libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
import joblib

In [3]:
log_data = pd.read_csv('/Users/raj/Desktop/CapStone/bankOps-platform/DataSets/application_logs.csv')

In [4]:
log_data.head()

,timestamp,log_level,log_message
0,2025-10-20 00:00:00,INFO,INFO: Transaction processed successfully. ID: ...
1,2025-10-20 00:01:00,ERROR,ERROR: NEFT transaction failed. Transaction ID...
2,2025-10-20 00:01:00,ERROR,ERROR: Database connection pool exhausted. Con...
3,2025-10-20 00:03:00,ERROR,ERROR: IMPS transaction failed. Transaction ID...
4,2025-10-20 00:03:00,WARNING,WARNING: Memory usage approaching threshold on...


In [5]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /Users/raj/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/raj/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/raj/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [6]:
log_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   timestamp    5000 non-null   object
 1   log_level    5000 non-null   object
 2   log_message  5000 non-null   object
dtypes: object(3)
memory usage: 117.3+ KB


In [7]:
log_data['timestamp'] = pd.to_datetime(
    log_data['timestamp']
)

In [8]:
def clean_log_message(message):

    # Convert text to lowercase
    message = message.lower()

    # Remove transaction IDs
    message = re.sub(
        r'txn\d+|transaction id[:\s]*\w+',
        ' ',
        message
    )

    # Remove IP addresses
    message = re.sub(
        r'\b(?:\d{1,3}\.){3}\d{1,3}\b',
        ' ',
        message
    )

    # Remove special characters
    message = re.sub(r'[^a-zA-Z0-9\s]', ' ', message)

    # Remove extra spaces
    message = re.sub(r'\s+', ' ', message).strip()

    return message

In [9]:
log_data['cleaned_message'] = (
    log_data['log_message'].astype(str).apply(clean_log_message)
)

In [10]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):

    words = text.split()

    filtered_words = [
        word for word in words
        if word not in stop_words
    ]

    return ' '.join(filtered_words)

In [11]:
log_data['cleaned_message'] = (
    log_data['cleaned_message'].apply(remove_stopwords)
)

In [12]:
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):

    words = text.split()

    lemmatized_words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return ' '.join(lemmatized_words)

In [13]:
log_data['cleaned_message'] = (
    log_data['cleaned_message'].apply(lemmatize_text)
)

In [14]:
tfidf = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95
)

tfidf_matrix = tfidf.fit_transform(
    log_data['cleaned_message']
)

In [15]:
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf.get_feature_names_out()
)

tfidf_df.head()

,01,01 current,01 latency,01 restarting,02,02 current,02 latency,02 restarting,03,03 current,...,usr847,usr882,usr956,utilization,utilization detected,warning,warning container,warning duplicate,warning memory,warning payment
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.15448,0.0,0.0,0.242511,0.0


In [16]:
log_level_encoded = pd.get_dummies(
    log_data['log_level'],
    prefix='log_level',
    dtype=int
)

In [17]:
final_log_features = pd.concat(
    [
        log_data[['timestamp']].reset_index(drop=True),
        log_level_encoded.reset_index(drop=True),
        tfidf_df.reset_index(drop=True)
    ],
    axis=1
)

final_log_features.head()

,timestamp,log_level_ERROR,log_level_INFO,log_level_WARNING,01,01 current,01 latency,01 restarting,02,02 current,...,usr847,usr882,usr956,utilization,utilization detected,warning,warning container,warning duplicate,warning memory,warning payment
0,2025-10-20 00:00:00,0,1,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0.0
1,2025-10-20 00:01:00,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0.0
2,2025-10-20 00:01:00,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0.0
3,2025-10-20 00:03:00,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0.0
4,2025-10-20 00:03:00,0,0,1,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.15448,0.0,0.0,0.242511,0.0


In [18]:
# Check TF-IDF matrix shape
print("TF-IDF Shape:", tfidf_matrix.shape)

TF-IDF Shape: (5000, 500)


In [28]:
scores = {}

for k in range(2, min(4, tfidf_matrix.shape[0])):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(tfidf_matrix)

    score = silhouette_score(
        tfidf_matrix,
        labels,
        metric='cosine'
    )

    scores[k] = score

    print(f"Clusters: {k}, Silhouette Score: {score:.4f}")

Clusters: 2, Silhouette Score: 0.1396
Clusters: 3, Silhouette Score: 0.1940


In [29]:
best_k = max(scores, key=scores.get)

print("Best number of clusters:", best_k)

Best number of clusters: 3


In [30]:
kmeans_model = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

log_data['cluster'] = kmeans_model.fit_predict(
    tfidf_matrix
)

log_data[['timestamp', 'log_level', 'log_message', 'cluster']].head(10)

,timestamp,log_level,log_message,cluster
0,2025-10-20 00:00:00,INFO,INFO: Transaction processed successfully. ID: ...,2
1,2025-10-20 00:01:00,ERROR,ERROR: NEFT transaction failed. Transaction ID...,0
2,2025-10-20 00:01:00,ERROR,ERROR: Database connection pool exhausted. Con...,0
3,2025-10-20 00:03:00,ERROR,ERROR: IMPS transaction failed. Transaction ID...,0
4,2025-10-20 00:03:00,WARNING,WARNING: Memory usage approaching threshold on...,1
5,2025-10-20 00:04:00,ERROR,ERROR: High CPU utilization detected on server...,1
6,2025-10-20 00:05:00,ERROR,ERROR: Customer authentication failed. User ID...,0
7,2025-10-20 00:06:00,WARNING,WARNING: Payment gateway response time exceede...,1
8,2025-10-20 00:08:00,INFO,INFO: Transaction processed successfully. ID: ...,2
9,2025-10-20 00:12:00,ERROR,ERROR: Disk I/O bottleneck detected on srv-18,1


In [31]:
cluster_mapping = {
    0: 'Normal Operations',
    1: 'Business KPI Issues',
    2: 'Infrastructure Issues'
}

In [ ]:
log_data['issue_category'] = log_data['cluster'].map(
    cluster_mapping
)

log_data[
    ['timestamp', 'log_level', 'log_message',
     'cluster', 'issue_category']
].head(10

,timestamp,log_level,log_message,cluster,issue_category
0,2025-10-20 00:00:00,INFO,INFO: Transaction processed successfully. ID: ...,2,Infrastructure Issues
1,2025-10-20 00:01:00,ERROR,ERROR: NEFT transaction failed. Transaction ID...,0,Normal Operations
2,2025-10-20 00:01:00,ERROR,ERROR: Database connection pool exhausted. Con...,0,Normal Operations
3,2025-10-20 00:03:00,ERROR,ERROR: IMPS transaction failed. Transaction ID...,0,Normal Operations
4,2025-10-20 00:03:00,WARNING,WARNING: Memory usage approaching threshold on...,1,Business KPI Issues
5,2025-10-20 00:04:00,ERROR,ERROR: High CPU utilization detected on server...,1,Business KPI Issues
6,2025-10-20 00:05:00,ERROR,ERROR: Customer authentication failed. User ID...,0,Normal Operations
7,2025-10-20 00:06:00,WARNING,WARNING: Payment gateway response time exceede...,1,Business KPI Issues
8,2025-10-20 00:08:00,INFO,INFO: Transaction processed successfully. ID: ...,2,Infrastructure Issues
9,2025-10-20 00:12:00,ERROR,ERROR: Disk I/O bottleneck detected on srv-18,1,Business KPI Issues


In [36]:


clustering_artifact = {
    'model': kmeans_model,
    'tfidf_vectorizer': tfidf,
    'cluster_mapping': cluster_mapping,
    'n_clusters': best_k
}

joblib.dump(
    clustering_artifact,
    'banking_log_clustering_model.pkl'
)

print("Log clustering model saved successfully!")

Log clustering model saved successfully!
